# 트랜스포머 (Transformer)

## RNN의 한계

> ### Attention의 등장

>> 1) Seq2Seq 복습
- 입력 시퀀스를 받아 다른 시퀀스로 변환하는 모델 구조
- 입력과 출력의 길이가 달라도 사용 가능 (ex. 영어 3단어 → 한국어 5단어)
- 번역, 요약, 음성 인식 등 다양한 변환 작업에 사용
- **Encoder**: LSTM이 단어를 순차적으로 읽은 뒤 문장 전체를 하나의 벡터로 압축
- **Context Vector**: 인코더가 만든 문장 요약본이자, 인코더와 디코더를 잇는 **유일한 통로**
- **Decoder**: Context Vector를 받아 번역문을 한 단어씩 생성

>> 2) Seq2Seq의 병목 현상
- Context Vector는 크기가 고정되어 있어, 같은 크기에 모든 정보를 담아야 함 → 문장이 길어질수록 **정보 손실**
- 디코더는 출력 단어마다 똑같은 요약본을 참고하므로, 단어별로 필요한 정보를 다르게 반영할 수 없음
- Context Vector는 인코더의 마지막 hidden state라서 앞쪽 단어 정보는 희미해짐

>> 3) Attention
- 정의: 현재 시점에서 가장 중요한 부분에 더 높은 가중치를 부여하는 기술
- 디코더가 단어를 생성할 때마다 입력 문장 전체를 다시 보고, 출력 단어마다 **새로운 Context Vector**를 만듦
- 과정
    1. 입력을 벡터로 변환
    2. 관련도 점수 계산 → Softmax로 합이 1인 비율(Attention Weight)로 변환
    3. 가중치만큼 각 단어의 hidden state를 섞어 Context Vector 생성
- 용어
    - Hidden state: RNN/LSTM이 단어를 읽을 때마다 갱신하는, 그 시점까지의 문맥 벡터
    - Attention Score: 디코더 현재 상태와 인코더 각 단어 사이의 관련도 점수
    - Attention Weight: Softmax를 거친 비율

>> 4) Attention의 장점
- 시간 경과에 대한 유연성 (vs. RNN): 거리와 상관없이 관계 파악 가능 (장기 의존성 문제 완화)
- 공간에 대한 유연성 (vs. CNN): 모든 위치를 직접 연결해 전역적 관계를 한 번에 학습
- **병렬화** (vs. RNN): 각 단어 쌍의 관련도 계산이 서로 독립적이라 동시 계산 가능
- 한계: 인코더·디코더가 여전히 LSTM 기반이라 모델 전체의 병렬화는 불가 → Attention은 RNN의 **보조 역할**


---

## Self-Attention

> ### Query, Key, Value

>> 1) 개념
- Attention = Query를 Key들과 비교해 관련도를 구하고, 그 비율만큼 Value를 섞어 결과를 만드는 계산
- **Query (Q)**: 현재 단어가 "지금 나한테 필요한 정보가 뭐지?"라고 묻는 벡터
- **Key (K)**: 각 단어가 "나는 이런 정보를 가지고 있다"고 보이는 이름표. Query와 비교해 관련도 결정
- **Value (V)**: 각 단어가 실제로 전달하는 정보. 관련도 비율만큼 섞여 최종 결과가 됨
- Key와 Value를 나누는 이유: "관련성 판단에 필요한 정보"와 "실제 전달할 정보"는 다를 수 있기 때문
- 도서관 비유: 질문(Q) → 책 제목표(K) 비교 → 책 내용(V)을 비율대로 섞어 답변

> ### Cross-Attention vs. Self-Attention

>> 1) Cross-Attention
- Query와 Key-Value가 **서로 다른 시퀀스**에서 나옴 (Q: 디코더 / K, V: 인코더)
- 디코더가 인코더에게 "지금 어디를 봐야 하지?"라고 묻는 구조 ("남에게 묻기")
- 오픈북 시험에서 교과서의 어디를 볼지 정하는 과정과 유사
- 한계: RNN 위에서 작동하므로 원문 내부 관계는 간접적으로만 파악, 병렬 처리 불가

>> 2) Self-Attention
- Query와 Key-Value가 **같은 시퀀스**에서 나옴 ("나에게 묻기")
- 예: "가을 은행 냄새 지독해"에서 '은행'이 '냄새'에 높은 가중치 → "가을에 냄새 나는 열매"로 의미 확정
- 모든 단어가 동시에 계산 → 병렬 처리 가능

- Transformer 내 사용 위치
    - 인코더: Self-Attention
    - 디코더: Self-Attention (뒤쪽 단어는 마스킹)
    - 인코더-디코더 사이: Cross-Attention

> ### Scaled Dot-Product Attention

>> 1) Dot-Product Attention
- Query와 Key를 **내적**해 관련도 점수를 구하는 방식 (방향이 비슷할수록 큰 값)
- 과정
    1. Q × Kᵀ → 모든 단어 쌍의 Attention Score 표 (n×n, 행렬 곱 한 번으로 동시 계산)
    2. 각 행에 Softmax → Attention Weight
    3. Attention Weight × V → 문맥이 반영된 새 벡터
- 문제점: 차원이 커지면 내적 값이 지나치게 커져 Softmax 확률이 한 단어에 쏠리고, 기울기가 0에 가까워져 학습이 어려워짐 (Vanishing Gradient)

>> 2) Scaled Dot-Product Attention
- 내적과 Softmax 사이에 **√d_k로 나누는 스케일링** 단계를 추가
- 분산이 1 근처로 유지되어 차원과 무관하게 안정적
- 효과 예시: 스케일링 전 sheepdog 82.2% → 후 64.5%로 완화, 주변 단어에도 비율 분배
- 수식

$$
Attention(Q,K,V)=softmax\left(\frac{QK^T}{\sqrt{d_K}}\right)V
$$

- 결과 벡터
    - Cross-Attention: Context Vector
    - Self-Attention: 문맥이 반영되어 의미가 업데이트된 단어 벡터

---

## Multi-Head Attention

> ### Single-Head Attention VS. Multi-Head Attention

>> 1) Head란?
- Attention 계산을 한 번 수행하는 독립된 단위
- 헤드마다 자기만의 가중치 행렬 W^Q, W^K, W^V를 가져, 같은 문장도 집중하는 단어가 다름

>> 2) Single-Head Attention
- 헤드 1개, 가중치 분포도 1개
- 한계: Softmax 가중치의 합은 1(100%)이라 여러 관계를 동시에 봐야 할 때 집중력을 나눠 써야 하고, 결과가 **흐릿한 벡터 하나**가 됨
- 예: "Tom gave Mary a book yesterday"에서 "gave"의 가중치가 Tom 0.25, Mary 0.25, book 0.25, yesterday 0.23으로 분산 → 누가 줬는지 불명확

>> 3) Multi-Head Attention
- Attention을 작게 쪼개 여러 번 동시에 수행
- 예: 헤드 1(Tom, Who?), 헤드 2(Mary, To whom?), 헤드 3(book, What?), 헤드 4(yesterday, When?) 각각 0.90 집중 → 정보가 섞이지 않고 또렷함
- 비유: 기자 1명(Single)이 전부 취재 vs. 기자 8명이 분야별로 취재하고 편집장(W^O)이 엮음(Multi)

> ### Multi-Head Attention 작동 방식

- 단어 512차원, 헤드 8개, 헤드당 64차원 (논문 기준)


- 헤드 내부 계산: 점수(Q·K 내적) → 스케일(÷√64=8) → Softmax → Value 가중합

---

## Transformer 전체 아키텍처

### 트랜스포머의 전처리 단계

- 흐름: 텍스트 입력 → 토큰화 → 임베딩 → Positional Encoding

> 1) 토큰화 (Tokenization)
- 문장을 모델이 다루는 기본 단위(토큰)로 자름
- 단어, 문장 부호가 토큰이 되고, 길거나 드문 단어는 여러 조각으로 분리 (`didn't` → `did` + `n't`)
- 논문: 영→독은 BPE(약 37,000개), 영→불은 word-piece(32,000개) 사전 사용

> 2) 임베딩 (Embedding)
- 각 토큰을 숫자 벡터로 변환 (벡터 길이 = 차원 d_model)
- 원리: **뜻이 비슷한 단어는 가까운 좌표**에 놓임 (처음엔 무작위, 학습하며 의미에 맞는 위치로 이동)
- (참고) Word2Vec: 다음 단어 예측 신경망의 마지막에서 두 번째 레이어 출력을 임베딩으로 사용

> 3) Positional Encoding
- Transformer는 모든 단어를 한꺼번에 보므로 순서를 모름 ("개는 고양이를 쫓는다" vs "고양이는 개를 쫓는다" 구분 불가)
- 위치마다 고정된 512차원 위치 벡터를 만들어 임베딩에 **더함** (이어 붙이지 않음)
- 문장이 달라도 같은 규칙을 적용하며, 같은 단어도 위치가 다르면 최종 벡터가 달라짐

### Encoder와 Decoder

> 1) Encoder
- 층 하나 = **Multi-Head Self-Attention** + **Feed-Forward Network**
- 단어를 한꺼번에 읽고, 출력은 단어별 문맥 벡터 전부 (RNN은 마지막 은닉 상태 하나)
- FFN
    - 수식: FFN(x) = max(0, xW₁ + b₁)W₂ + b₂
    - 512 → 2048 → 512, 사이에 ReLU
    - 단어마다 따로, 같은 가중치로 적용 (단어 간 정보 교환 없음)
    - 비유: Self-Attention = 조별 토론, FFN = 1:1 첨삭
- **Add & Norm**
    - Add (Residual Connection): 부품 출력에 원래 입력을 더함 (원본 보존, 변경분만 덧붙임)
    - Norm (Layer Normalization): 한 단어 벡터의 512개 값을 평균 0, 분산 1 근처로 맞춤 (학습 안정화)

> 2) Decoder
- 층 하나 = 세 부품 (각각 뒤에 Add & Norm)
    1. **Masked Multi-Head Self-Attention**: 지금까지 쓴 단어끼리 관계 파악, 미래 단어는 가림
    2. **Encoder-Decoder Attention (Cross-Attention)**: 원문 분석 결과에서 필요한 부분 탐색
    3. **Feed-Forward Network**: 인코더와 동일
- 6층 통과 후 Linear + Softmax로 다음 단어 확률 계산
- Masked Attention
    - 미래 단어를 가려(Softmax 후 0) 앞의 단어만 보게 함
    - 학습 시 정답 문장 전체를 병렬 입력하므로, 마스크가 없으면 "컨닝"이 됨
- Cross-Attention의 Q·K·V 출처
    - Q: 직전 Decoder 서브레이어의 출력
    - K, V: Encoder 마지막 층의 출력 (6개 디코더 층이 모두 동일한 출력 사용)

> 3) 전체 데이터 흐름 ("I love you" 번역)
- 입력 준비: 임베딩 + Positional Encoding으로 [의미 + 위치] 벡터 생성
- 인코더: 6개 층에서 Self-Attention과 FFN으로 문맥 이해 → 출력이 Decoder의 K·V로 전달
- 디코더: 한 단어씩 생성